In [2]:
"""
undetected_bc_across_models.py
===============================
Lists custom_ids where the breaking change (BC) was detected or NOT detected
across different models and context variants.

Outputs:
  1. undetected_bc_across_models.csv  — long format, undetected instances
  2. undetected_bc_summary.csv        — one row per custom_id, undetected summary
  3. detected_bc_across_models.csv    — long format, detected instances
  4. detected_bc_summary.csv          — one row per custom_id, detected summary
"""

import json
import csv
from pathlib import Path
from collections import defaultdict


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG — UPDATE THESE BEFORE EACH RUN
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Each entry: (model_name, context_variant, path_to_transplant_results_json)
LLM_CONFIGS = [
    # --- GPT-4o ---
    ("GPT-4o", "Minimal", "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPT-4o", "Method",  "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPT-4o", "Class",   "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json"),

    # --- Qwen-480B ---
    ("Qwen-480B", "Minimal", "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("Qwen-480B", "Method",  "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("Qwen-480B", "Class",   "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"),

    # --- GPTOSS-120b ---
    ("GPTOSS-120b", "Minimal", "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPTOSS-120b", "Method",  "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"),
    ("GPTOSS-120b", "Class",   "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"),
]

# BUMP CSV for ground-truth metadata
BUMP_CSV = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes.csv"

# Output files
OUTPUT_CSV           = "/Volumes/Rachna-HD/RQResultsForPaper/RQ2/Stats/undetected_bc_across_models.csv"
SUMMARY_CSV          = "/Volumes/Rachna-HD/RQResultsForPaper/RQ2/Stats/undetected_bc_summary.csv"
DETECTED_CSV         = "/Volumes/Rachna-HD/RQResultsForPaper/RQ2/Stats/detected_bc_across_models.csv"
DETECTED_SUMMARY_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ2/Stats/detected_bc_summary.csv"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# END CONFIG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


def parse_bump_errors(raw: str) -> str:
    if not raw or str(raw).strip() in ('', 'nan'):
        return ''
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        result.add(e.split('.')[-1])
    return '|'.join(sorted(result))


def load_bump(bump_csv: str) -> dict:
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            cid = row['custom_id']
            data[cid] = {
                'bump_bc_errors':      parse_bump_errors(row.get('exception_types', '')),
                'bump_bc_errors_raw':  row.get('exception_types', ''),
                'failure_category':    row.get('failureCategory', ''),
            }
    return data


def load_llm_results(llm_json: str) -> dict:
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    print("Loading BUMP data...")
    bump = load_bump(BUMP_CSV)
    print(f"  BUMP instances: {len(bump)}")

    undetected_rows = []
    detected_rows = []
    undetected_tracker = defaultdict(set)  # custom_id → set of (model, variant)
    detected_tracker   = defaultdict(set)  # custom_id → set of (model, variant)
    total_configs = len(LLM_CONFIGS)

    for model_name, context_variant, json_path in LLM_CONFIGS:
        print(f"\nProcessing: {model_name} / {context_variant}")
        print(f"  File: {json_path}")

        if not Path(json_path).is_file():
            print(f"  WARNING: File not found — skipping")
            continue

        llm = load_llm_results(json_path)
        print(f"  Instances loaded: {len(llm)}")

        total_instances = 0
        detected_count = 0
        undetected_count = 0

        for instance_id, instance_data in llm.items():
            total_instances += 1
            tests_data   = instance_data.get('tests', {})
            passed_files = tests_data.get('passed', [])
            failed_tests = tests_data.get('failed', [])

            total_tests  = len(passed_files) + len(failed_tests)
            num_detected = len(failed_tests)
            num_missed   = len(passed_files)

            bc_detected = num_detected > 0
            bump_info   = bump.get(instance_id, {})

            row = {
                'custom_id':                instance_id,
                'model':                    model_name,
                'context_variant':          context_variant,
                'total_tests_generated':    total_tests,
                'tests_passed_v2':          num_missed,
                'tests_failed_v2':          num_detected,
                'bc_detected':              bc_detected,
                'bump_bc_failure_category': bump_info.get('failure_category', ''),
                'bump_bc_errors':           bump_info.get('bump_bc_errors', ''),
                'bump_bc_errors_raw':       bump_info.get('bump_bc_errors_raw', ''),
            }

            if bc_detected:
                detected_count += 1
                detected_tracker[instance_id].add((model_name, context_variant))
                detected_rows.append(row)
            else:
                undetected_count += 1
                undetected_tracker[instance_id].add((model_name, context_variant))
                undetected_rows.append(row)

        print(f"  BC detected:   {detected_count} / {total_instances}")
        print(f"  BC undetected: {undetected_count} / {total_instances}")

    # ── Long-format fieldnames (shared) ───────────────────────────────────────
    long_fieldnames = [
        'custom_id', 'model', 'context_variant',
        'total_tests_generated', 'tests_passed_v2', 'tests_failed_v2', 'bc_detected',
        'bump_bc_failure_category', 'bump_bc_errors', 'bump_bc_errors_raw',
    ]

    # ── Write undetected long-format CSV ──────────────────────────────────────
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=long_fieldnames)
        writer.writeheader()
        writer.writerows(undetected_rows)

    # ── Write undetected summary CSV ──────────────────────────────────────────
    Path(SUMMARY_CSV).parent.mkdir(parents=True, exist_ok=True)
    undetected_summary_rows = []
    for cid, configs in sorted(undetected_tracker.items()):
        bump_info = bump.get(cid, {})
        models_missed   = sorted(set(m for m, v in configs))
        variants_missed = sorted(set(v for m, v in configs))
        undetected_summary_rows.append({
            'custom_id':                 cid,
            'num_configs_undetected':    len(configs),
            'total_configs':             total_configs,
            'undetected_in_all_configs': len(configs) == total_configs,
            'models_undetected':         '|'.join(models_missed),
            'variants_undetected':       '|'.join(variants_missed),
            'model_variant_pairs':       '|'.join(f"{m}+{v}" for m, v in sorted(configs)),
            'bump_bc_failure_category':  bump_info.get('failure_category', ''),
            'bump_bc_errors':            bump_info.get('bump_bc_errors', ''),
        })
    undetected_summary_rows.sort(key=lambda r: (-r['num_configs_undetected'], r['custom_id']))

    undetected_summary_fields = [
        'custom_id', 'num_configs_undetected', 'total_configs', 'undetected_in_all_configs',
        'models_undetected', 'variants_undetected', 'model_variant_pairs',
        'bump_bc_failure_category', 'bump_bc_errors',
    ]
    with open(SUMMARY_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=undetected_summary_fields)
        writer.writeheader()
        writer.writerows(undetected_summary_rows)

    # ── Write detected long-format CSV ────────────────────────────────────────
    Path(DETECTED_CSV).parent.mkdir(parents=True, exist_ok=True)
    with open(DETECTED_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=long_fieldnames)
        writer.writeheader()
        writer.writerows(detected_rows)

    # ── Write detected summary CSV ────────────────────────────────────────────
    Path(DETECTED_SUMMARY_CSV).parent.mkdir(parents=True, exist_ok=True)
    detected_summary_rows = []
    for cid, configs in sorted(detected_tracker.items()):
        bump_info = bump.get(cid, {})
        models_detected   = sorted(set(m for m, v in configs))
        variants_detected = sorted(set(v for m, v in configs))
        detected_summary_rows.append({
            'custom_id':                cid,
            'num_configs_detected':     len(configs),
            'total_configs':            total_configs,
            'detected_in_all_configs':  len(configs) == total_configs,
            'models_detected':          '|'.join(models_detected),
            'variants_detected':        '|'.join(variants_detected),
            'model_variant_pairs':      '|'.join(f"{m}+{v}" for m, v in sorted(configs)),
            'bump_bc_failure_category': bump_info.get('failure_category', ''),
            'bump_bc_errors':           bump_info.get('bump_bc_errors', ''),
        })
    detected_summary_rows.sort(key=lambda r: (-r['num_configs_detected'], r['custom_id']))

    detected_summary_fields = [
        'custom_id', 'num_configs_detected', 'total_configs', 'detected_in_all_configs',
        'models_detected', 'variants_detected', 'model_variant_pairs',
        'bump_bc_failure_category', 'bump_bc_errors',
    ]
    with open(DETECTED_SUMMARY_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=detected_summary_fields)
        writer.writeheader()
        writer.writerows(detected_summary_rows)

    # ── Console summary ───────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"BC DETECTION SUMMARY ACROSS MODELS")
    print(f"{'='*70}")
    print(f"Total model/variant configs:         {total_configs}")

    print(f"\n--- Undetected ---")
    print(f"Total undetected rows (long format): {len(undetected_rows)}")
    print(f"Unique custom_ids with ≥1 miss:      {len(undetected_tracker)}")
    all_missed = sum(1 for cfgs in undetected_tracker.values() if len(cfgs) == total_configs)
    print(f"Undetected in ALL configs:           {all_missed}")
    print(f"Undetected in SOME configs:          {len(undetected_tracker) - all_missed}")

    print(f"\n--- Detected ---")
    print(f"Total detected rows (long format):   {len(detected_rows)}")
    print(f"Unique custom_ids detected ≥1:       {len(detected_tracker)}")
    all_detected = sum(1 for cfgs in detected_tracker.values() if len(cfgs) == total_configs)
    print(f"Detected in ALL configs:             {all_detected}")
    print(f"Detected in SOME configs:            {len(detected_tracker) - all_detected}")

    print(f"\nPer-config counts:")
    print(f"  {'Model':<20s} {'Variant':<10s} {'Detected':>10s} {'Undetected':>12s}")
    print(f"  {'-'*20} {'-'*10} {'-'*10} {'-'*12}")
    for model_name, context_variant, _ in LLM_CONFIGS:
        det   = sum(1 for cfgs in detected_tracker.values() if (model_name, context_variant) in cfgs)
        undet = sum(1 for cfgs in undetected_tracker.values() if (model_name, context_variant) in cfgs)
        print(f"  {model_name:<20s} {context_variant:<10s} {det:>10d} {undet:>12d}")

    print(f"\nOutput written to:")
    print(f"  Undetected long-format: {OUTPUT_CSV}")
    print(f"  Undetected summary:     {SUMMARY_CSV}")
    print(f"  Detected long-format:   {DETECTED_CSV}")
    print(f"  Detected summary:       {DETECTED_SUMMARY_CSV}")


if __name__ == "__main__":
    main()

Loading BUMP data...
  BUMP instances: 89

Processing: GPT-4o / Minimal
  File: /Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json
  Instances loaded: 38
  BC detected:   17 / 38
  BC undetected: 21 / 38

Processing: GPT-4o / Method
  File: /Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json
  Instances loaded: 32
  BC detected:   13 / 32
  BC undetected: 19 / 32

Processing: GPT-4o / Class
  File: /Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json
  Instances loaded: 31
  BC detected:   27 / 31
  BC undetected: 4 / 31

Processing: Qwen-480B / Minimal
  File: /Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json
  Instances loaded: 32
  BC detected:   18 / 32
  BC undetected: 14 / 32

Processing: Qwen-480B / Method
  File: /Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/transplant_results_br